# Binance Spot Historical Data Import

This notebook imports historical OHLCV data from Binance Spot markets into the PostgreSQL database.

**Data Source:** Binance Public Data (https://data.binance.vision/)

**Features:**
- Downloads historical klines/candlestick data directly from Binance
- Supports multiple timeframes (1m, 5m, 15m, 1h, 4h, 1d)
- Automatic duplicate detection
- Bulk insert operations

**Data Format:**
- Binance CSV: [Open time, Open, High, Low, Close, Volume, Close time, Quote volume, Trades, Taker buy base, Taker buy quote, Ignore]
- Timestamps are in milliseconds (UNIX epoch)

**API Endpoints:**
- Monthly data: `https://data.binance.vision/data/spot/monthly/klines/{SYMBOL}/{INTERVAL}/{SYMBOL}-{INTERVAL}-{YEAR}-{MONTH}.zip`
- Daily data: `https://data.binance.vision/data/spot/daily/klines/{SYMBOL}/{INTERVAL}/{SYMBOL}-{INTERVAL}-{YEAR}-{MONTH}-{DAY}.zip`

In [ ]:
import requests
import pandas as pd
import zipfile
from io import BytesIO
from datetime import datetime, timedelta


from clients.db_client import DBClient

In [ ]:
# Configuration
BASE_URL = "https://data.binance.vision/data/spot/monthly/klines"
INTERVAL = "1h"  # Options: 1m, 5m, 15m, 30m, 1h, 4h, 1d, etc.
START_YEAR = 2023
START_MONTH = 1

In [ ]:
def download_binance_klines(symbol: str, interval: str, year: int, month: int) -> pd.DataFrame:
    """
    Download monthly klines data from Binance public repository.
    """
    month_str = f"{month:02d}"
    url = f"{BASE_URL}/{symbol}/{interval}/{symbol}-{interval}-{year}-{month_str}.zip"
    
    print(f"Downloading: {symbol} {year}-{month_str}...", end=" ")
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        with zipfile.ZipFile(BytesIO(response.content)) as zf:
            csv_filename = f"{symbol}-{interval}-{year}-{month_str}.csv"
            with zf.open(csv_filename) as csv_file:
                df = pd.read_csv(csv_file, names=[
                    'open_time', 'open', 'high', 'low', 'close', 'volume',
                    'close_time', 'quote_volume', 'trades', 'taker_buy_base',
                    'taker_buy_quote', 'ignore'
                ])
                
                # Auto-detect unit: microseconds (16 digits) or milliseconds (13 digits)
                first_timestamp = df['open_time'].iloc[0]
                if first_timestamp > 1e15:  # More than 15 digits = microseconds
                    df['timestamp'] = pd.to_datetime(df['open_time'], unit='us', utc=True)
                else:  # milliseconds
                    df['timestamp'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
                
                df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume']]
                
                for col in ['open', 'high', 'low', 'close', 'volume']:
                    df[col] = df[col].astype(float)
                
                print(f"{len(df)} rows")
                return df
                
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            print("Not available")
        else:
            print(f"HTTP Error: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

In [ ]:
def generate_months(start_year: int, start_month: int, end_year: int = 2025, end_month: int = 12):
    """
    Generate year-month pairs from start date to end date.
    
    Args:
        start_year: Starting year
        start_month: Starting month (1-12)
        end_year: Ending year (default: 2025)
        end_month: Ending month (default: 12)
    """
    start = datetime(start_year, start_month, 1)
    end = datetime(end_year, end_month, 1)
    
    months = []
    while start <= end:
        months.append((start.year, start.month))
        # Move to next month
        if start.month == 12:
            start = datetime(start.year + 1, 1, 1)
        else:
            start = datetime(start.year, start.month + 1, 1)
    
    return months

In [ ]:
with DBClient() as db:
    # Import Binance data
    total_inserted = 0
    total_skipped = 0

    # Get all Binance instruments from database
    instruments_dict = db.get_instruments_by_exchange(exchange="binance")
    print(f"Found {len(instruments_dict)} Binance instruments in database\n")

    # Generate list of months to download
    months = generate_months(START_YEAR, START_MONTH)

    for symbol, instrument_id in instruments_dict.items():
        print(f"\n=== Processing {symbol} ===")
        
        # Download each month
        for year, month in months:
            df = download_binance_klines(symbol, INTERVAL, year, month)
            
            if df.empty:
                continue
            
            # Get existing timestamp range from database
            db_min, db_max = db.get_timestamp_range(instrument_id)
            
            csv_start = df['timestamp'].min()
            csv_end = df['timestamp'].max()
            
            # Check for duplicates
            if db_min and db_max:
                # Filter out overlapping data
                df = df[(df['timestamp'] < db_min) | (df['timestamp'] > db_max)]
                
                if df.empty:
                    print(f"All data already exists, skipping")
                    total_skipped += 1
                    continue
            
            # Insert data
            df['instrument_id'] = instrument_id
            rows = db.insert_market_data(df)
            total_inserted += rows
            print(f"Inserted {rows} rows")

    print(f"Total rows inserted: {total_inserted}")
    print(f"Total months skipped (duplicates): {total_skipped}")